In [4]:
"""
FOFF Simulation Core
Paper Title: FOFF: An Energy-Efficient Task Offloading in VEC-Enabled Vehicular Networks Using Fuzzy
TOPSIS
Author: Antonio S. S. Vieira (sergio.vieira@ifce.edu.br) - 2025-01
"""

import logging
import random
import numpy as np
import copy
from offloading_simulator import *
from random_strategy import RandomStrategy
from gtt_strategy import GTTStrategy
from gcf_strategy import GCFStrategy
from foff_strategy import FOFFStrategy
from foff_strategy import extract_nft_weighted, extract_nft_map
from config import configure_logging
from plotting import *
import hashlib
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from file_utils import *
from task_manager import parse_task_string


def hash_from_array(arr):
    array_string = str(arr).encode('utf-8')
    return hashlib.md5(array_string).hexdigest()

# Aqui os 6 dispositivos e suas caracteísticas
# Cada dispositivo possui:
#   Frequencia do cpu em GHZ (f)
#   Coeficiente de custo energético nj/ciclo (kappa)
#   Fila de espera segundos (Q)
#   Coordenadas (x, y) para cálculo de distância. Em metros. (position)
#   Identificador (id)
# (f, kappa, Q, pos, id) 
CHALLEGING_SCENARIO = [
    (2, 90, 0.0, np.array([451.36, 741.45]), 0),
    (4,  670, 0.0, np.array([642.61, 697.24]), 1),
    (2, 5, 0.0, np.array([722.94, 603.27]), 2),
    (4, 345, 0.0, np.array([670.94, 318.01]), 3),    
    (2, 88, 0.0, np.array([256.37, 503.55]), 4),    
    (1, 670, 900.0, np.array([500.0, 500.0]), 5),
]
CHALLEGING_SCENARIO_HASH = hash_from_array(CHALLEGING_SCENARIO)

# (f, kappa, Q, pos, id) 
BALANCED_SCENARIO = [
    (2, 5, 0.0, np.array([451.36, 741.45]), 0),
    (3, 11, 0.0, np.array([256.37, 503.55]), 1),
    (2, 10, 0.0, np.array([722.94, 603.27]), 2),
    (4, 23, 0.0, np.array([670.94, 318.01]), 3),
    (2, 12, 0.0, np.array([642.61, 697.24]), 4),
    (1, 20, 900.0, np.array([500.0, 500.0]), 5),    
]
BALANCED_SCENARIO_HASH = hash_from_array(CHALLEGING_SCENARIO)

CURRENT_SCENARIO = CHALLEGING_SCENARIO
CURRENT_SCENARIO_HASH = hash_from_array(CURRENT_SCENARIO)


# -------------------- Main Execution --------------------
logger = configure_logging(level=logging.DEBUG)
config = SimulationConfig()
config.num_tasks = 1
config.total_tests = 1
random.seed(config.seed)

# OBUI está sempre na última posição da lista de dispositivos
obui_index = CURRENT_SCENARIO[len(CURRENT_SCENARIO) - 1][4]
sim = OffloadingSimulator(config, obui_index)
# original_tasks = list(sim._generate_tasks(config.num_tasks))
# tasks_shuffled = [random.sample(original_tasks, len(original_tasks)) for _ in range(config.total_tests)]
# tasks_shuffled = [[Task(data_size=993, cpu_cycles=11, task_id=0, arrival_time=699.2756195608383)]]

tasks_shuffled = load("/home/sergiosvieira/projects/foff/tasks_shuffled.csv")
for i in range(len(tasks_shuffled)):
    for j in range(len(tasks_shuffled[0])):
        tasks_shuffled[i][j] = parse_task_string(tasks_shuffled[i][j])

strategies = {
    'FOFF': (FOFFStrategy(sim, ['m', 'm', extract_nft_map]), []),
    # 'ROFF': (RandomStrategy(sim), []),
    # 'GTT' : (GTTStrategy(sim), []),
    # 'GCF' : (GCFStrategy(sim), []),
}
columns = list(strategies.keys())

import itertools

def create_combination_matrix(importances):
    combinations = list(itertools.product(importances, repeat=2))
    matrix = []
    for i in range(0, len(combinations), len(importances)):
        matrix.append(combinations[i:i + len(importances)])
    return matrix

importances = ["vl", "l", "ml", "m", "mh", "h", "vh"]
combination_matrix = create_combination_matrix(importances)
# Run all simulations and collect results
for row in combination_matrix:
    for importances_labels in row:
        for key, (off_strategy, all_results) in strategies.items():       
            for counter, tasks in enumerate(tasks_shuffled):
                sim.set_devices(copy.deepcopy(CURRENT_SCENARIO))
                # logger.debug(f"===== Test {counter + 1} - {importances_labels} - {tasks} =====")
                if off_strategy.__str__() == "FOFF":
                    off_strategy.set_omega(importances_labels)
                result = sim.run_simulation(tasks, off_strategy, sensitivity_test=True)
                all_results.append(result)
            
# Aggregate energy and latency data for all strategies
total_energy_data = []
total_latency_data = []
total_cycles_data = []
total_select_data = []
for key, (_, all_results) in strategies.items():
    results = all_results
    total_energy_data.append([result.total_energy for result in results])
    total_latency_data.append([result.total_latency for result in results])
    total_cycles_data.append([result.processed_cycles for result in results])
    total_select_data.append([result.device_selections for result in results])

# Convert to numpy arrays
total_energy_data = np.array(total_energy_data)
total_latency_data = np.array(total_latency_data)
total_cycles_data = np.array(total_cycles_data)
# Plotting
scenario_str = "Challenging" if CURRENT_SCENARIO_HASH == CHALLEGING_SCENARIO_HASH else "Balanced"
plot_bar_with_ci(total_energy_data, columns, confidence=0.95, 
                 title=f"Total Energy Consumption x Algorithm (CI 95%) - {scenario_str} Scenario", 
                 ylabel="Total Energy Consumption (Joules)")
plot_bar_with_ci(total_latency_data, columns, 0.95,
                 title=f"Total Processed Time x Algorithm (CI 95%) - {scenario_str} Scenario", 
                 ylabel="Total Processed Time (Seconds)")
plot_bar_with_ci(total_cycles_data, columns, 0.95,
                 title=f"Total Processed Cycles x Algorithm (CI 95%) - {scenario_str} Scenario", 
                 ylabel="Total Processed Cycles")
plot_device_selection_count(total_select_data, columns, CURRENT_SCENARIO)

FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF
FOFF


Traceback (most recent call last):
  File "/home/sergiosvieira/.local/lib/python3.13/site-packages/IPython/core/interactiveshell.py", line 3577, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1582/2741589584.py", line 112, in <module>
    result = sim.run_simulation(tasks, off_strategy, sensitivity_test=True)
  File "/home/sergiosvieira/projects/foff/source-code/offloading_simulator.py", line 88, in run_simulation
    selected_device, ranking = offloading_strategy.execute(tasks, task.task_id, self.device_manager)
                               ~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/sergiosvieira/projects/foff/source-code/foff_strategy.py", line 103, in execute
    results = self.calculate(task, device_manager.devices)
  File "/home/sergiosvieira/projects/foff/source-code/foff_strategy.py", line 97, in calculate
    total_latency, total_energy = se

In [ ]:
import matplotlib.pyplot as plt
import math

def plot_device_scores_separately(all_results):
    import math
    import matplotlib.pyplot as plt

    device_ids = ['0','1','2','3','4','5']
    category_pairs = []
    device_scores = {dev: [] for dev in device_ids}

    # Percorre os resultados e guarda as pontuações de cada dispositivo
    for result in all_results:
        # Se ranking estiver vazio, pula este resultado
        if not result.ranking:
            continue

        ((cat1, cat2), disp_scores) = result.ranking[0]
        category_pairs.append((cat1, cat2))

        # Exemplo: { '3': 0.9453, '2': 0.9051, ... }
        scores_map = {d: s for d, s in disp_scores}

        # Para cada dispositivo, pega a pontuação. Se não existir, 0.0
        for dev in device_ids:
            device_scores[dev].append(scores_map.get(dev, 0.0))

    # Se, ao fim, não tivermos pares de categoria, saímos
    if not category_pairs:
        print("Não há dados para plotar (category_pairs está vazio).")
        return

    n = len(device_ids)
    cols = 3
    rows = math.ceil(n / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(14, 7), sharey=True)
    axes = axes.flatten()

    x = range(len(category_pairs))
    x_labels = [f"{cat1}-{cat2}" for (cat1, cat2) in category_pairs]

    for i, dev in enumerate(device_ids):
        ax = axes[i]
        ax.plot(x, device_scores[dev], marker='o', label=f"Dev {dev}")
        ax.set_title(f"Dispositivo {dev}")
        ax.set_xticks(x)
        ax.set_xticklabels(x_labels, rotation=45, ha='right')
        ax.set_xlabel("Par de Categorias")
        ax.set_ylabel("Pontuação")
        ax.legend()

    # Caso sobrem eixos não usados
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.show()


plot_device_scores_separately(all_results)

In [ ]:
import math
import matplotlib.pyplot as plt

def plot_device_rank_positions_separately(all_results):
    device_ids = ['0','1']
    category_pairs = []
    # Em vez de device_scores, agora armazenaremos device_positions
    device_positions = {dev: [] for dev in device_ids}

    for result in all_results:
        # Se ranking estiver vazio, apenas ignoramos
        if not result.ranking:
            continue

        ((cat1, cat2), disp_scores) = result.ranking[0]
        category_pairs.append((cat1, cat2))

        # disp_scores é algo como: [['3', 0.94...], ['2', 0.90...], ...]
        # Queremos mapear cada dispositivo à sua posição (1 = melhor)
        rank_map = {}
        for i, (dev, _) in enumerate(disp_scores):
            rank_map[dev] = i + 1  # i+1 = posição 1-based

        # Agora armazenamos a posição para cada dispositivo
        for dev in device_ids:
            # Se não existir no rank_map, retornamos None ou 0
            device_positions[dev].append(rank_map.get(dev, None))

    # Caso não haja pares de categoria, não há o que plotar
    if not category_pairs:
        print("Não há dados para plotar (category_pairs está vazio).")
        return

    # Organizar a figura em subplots
    n = len(device_ids)
    cols = 1
    rows = math.ceil(n / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(14, 14), sharey=True)
    axes = axes.flatten()  # Transforma numa lista para iterar

    x = range(len(category_pairs))
    x_labels = [f"{cat1}-{cat2}" for (cat1, cat2) in category_pairs]

    for i, dev in enumerate(device_ids):
        ax = axes[i]
        ax.plot(x, device_positions[dev], marker='o', label=f"Dev {dev}")
        ax.set_title(f"Device {dev:02}")
        ax.set_xticks(x)
        ax.set_xticklabels(x_labels, rotation=45, ha='right')
        ax.set_xlabel("Weights")
        ax.set_ylabel("Ranking Position")

        # Inverte o eixo Y para que posição 1 fique no topo
        ax.invert_yaxis()

        ax.legend()

    # Se houver subplots excedentes, desativá-los
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.show()

plot_device_rank_positions_separately(all_results)

In [ ]:
def plot_heatmaps_in_groups(all_results, group_size=7):
    """
    Recebe all_results, agrupa as combinações de 7 em 7 e plota os heatmaps em duas colunas.

    Parâmetros:
      - all_results: lista de resultados, onde cada resultado possui um atributo 'ranking'
        com a estrutura ((cat1, cat2), disp_scores).
      - group_size: número de combinações por grupo (padrão = 7).
    """
    # Extrai os dados e os rótulos de cada combinação
    data = []
    labels_linhas = []
    for result in all_results:
        ((cat1, cat2), disp_scores) = result.ranking[0]
        # Usa 'd' convertido para inteiro
        data.append([float(s) for d, s in disp_scores])
        labels_linhas.append(f"{cat1}-{cat2}")
    
    # Agrupa os dados e os rótulos em blocos de tamanho group_size
    groups_data = [data[i:i+group_size] for i in range(0, len(data), group_size)]
    groups_labels = [labels_linhas[i:i+group_size] for i in range(0, len(labels_linhas), group_size)]
    
    num_groups = len(groups_data)
    
    # Define o layout em duas colunas
    n_cols = 2
    n_rows = math.ceil(num_groups / n_cols)
    
    # Cria a figura e os eixos
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 6 * n_rows))
    
    # Garante que axes seja um array 1D para iteração (mesmo se for apenas uma linha)
    if n_rows * n_cols > 1:
        axes = axes.flatten()
    else:
        axes = [axes]
    
    # Rótulos das colunas (dispositivos)
    labels_colunas = ["Edge 00", "Edge 01", "Edge 02", "Edge 03", "Edge 04", "OBUI"]
    
    # Plota cada heatmap em seu respectivo subplot
    for ax, group_data, group_labels in zip(axes, groups_data, groups_labels):
        sns.heatmap(
            group_data,
            ax=ax,
            annot=True,
            fmt="f",          # Mostra valores inteiros
            cmap="coolwarm",
            vmin=0,
            vmax=1,
            cbar_kws={"label": "CCi"},
            xticklabels=labels_colunas,
            yticklabels=group_labels
        )
        ax.set_xlabel("Devices")
        ax.set_ylabel("Importance Weight Combinations")
        ax.set_title("Heatmap de Ranks")
    
    # Remove os eixos não utilizados (caso o grid tenha mais subplots que grupos)
    for i in range(num_groups, len(axes)):
        fig.delaxes(axes[i])
    
    plt.tight_layout()
    plt.show()

plot_heatmaps_in_groups(all_results)

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

num_combinacoes = 49
num_dispositivos = 6

data = []
labels_linhas = []
for result in all_results:
    ((cat1, cat2), disp_scores) = result.ranking[0]
    data.append([float(s) for d, s in disp_scores])
    labels_linhas.append(f"{cat1}-{cat2}")

labels_colunas = ["Edge 00", "Edge 01", "Edge 02", "Edge 03", "Edge 04", "OBUI"]

plt.figure(figsize=(8, 6))
ax = sns.heatmap(
    data,
    annot=False,
    fmt="f",          # "d" para mostrar valores inteiros
    cmap="coolwarm",
    vmin=0,
    vmax=1,
    cbar_kws={"label": "CCi"},  # Rótulo da barra de cores
    xticklabels=labels_colunas,
    yticklabels=labels_linhas
)

# Ajusta os rótulos dos eixos
ax.set_xlabel("Devices")
ax.set_ylabel("Importance Weight Combinations")
plt.title("Heatmap de Ranks")

# Exibe o gráfico
plt.tight_layout()
plt.show()


In [ ]:
def plot_sensitivity_bar(data_scenario1, data_scenario2, devices):
    """
    Cria um gráfico de barras comparando o desvio-padrão de cada dispositivo
    nos dois cenários.
    """
    # Converta em arrays NumPy, caso ainda não sejam
    data_s1 = np.array(data_scenario1)
    data_s2 = np.array(data_scenario2)
    
    # Calcula o desvio-padrão de cada coluna (dispositivo) ao longo das combinações
    stds_s1 = data_s1.std(axis=0)
    stds_s2 = data_s2.std(axis=0)
    
    # Cria um eixo X para 6 dispositivos
    x = np.arange(len(devices))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(8,5))
    
    # Barras para cenário 1 (deslocadas levemente para a esquerda)
    rects1 = ax.bar(x - width/2, stds_s1, width, label='DirectMapping')
    
    # Barras para cenário 2 (deslocadas levemente para a direita)
    rects2 = ax.bar(x + width/2, stds_s2, width, label='WeightedMapping')
    
    ax.set_ylabel('Standard Deviation of Rank')
    ax.set_xticks(x)
    ax.set_xticklabels(devices)
    ax.legend()
    ax.set_title('Comparação de Sensibilidade (Desvio-Padrão)')

    plt.tight_layout()
    plt.show()

data_map = [[5, 0, 1, 2, 3, 4], [5, 1, 4, 0, 2, 3], [5, 1, 3, 4, 0, 2], [4, 0, 2, 5, 3, 1], [0, 2, 4, 5, 1, 3], [5, 0, 1, 4, 2, 3], [0, 1, 3, 4, 5, 2], [0, 2, 4, 1, 3, 5], [0, 1, 3, 4, 5, 2], [0, 4, 5, 1, 2, 3], [2, 4, 5, 0, 3, 1], [5, 0, 1, 3, 4, 2], [0, 2, 3, 4, 5, 1], [5, 1, 2, 3, 4, 0], [1, 2, 4, 3, 0, 5], [2, 3, 4, 0, 1, 5], [0, 3, 4, 5, 1, 2], [4, 1, 2, 3, 5, 0], [0, 1, 2, 3, 5, 4], [5, 0, 2, 3, 4, 1], [5, 2, 3, 4, 0, 1], [1, 3, 4, 0, 2, 5], [0, 1, 2, 4, 3, 5], [0, 1, 2, 4, 3, 5], [1, 4, 5, 0, 2, 3], [0, 1, 2, 4, 5, 3], [0, 5, 1, 2, 3, 4], [0, 1, 2, 5, 3, 4], [0, 1, 2, 3, 4, 5], [1, 4, 0, 2, 3, 5], [0, 1, 2, 4, 3, 5], [2, 0, 1, 3, 4, 5], [2, 1, 3, 4, 5, 0], [1, 2, 4, 0, 5, 3], [4, 2, 3, 1, 5, 0], [2, 4, 1, 3, 0, 5], [1, 2, 4, 0, 3, 5], [0, 1, 3, 2, 4, 5], [1, 2, 4, 0, 3, 5], [1, 0, 2, 3, 4, 5], [0, 3, 4, 1, 5, 2], [1, 3, 4, 2, 5, 0], [0, 1, 2, 3, 4, 5], [0, 1, 2, 3, 4, 5], [0, 3, 4, 2, 1, 5], [1, 2, 4, 0, 3, 5], [1, 3, 0, 2, 4, 5], [2, 3, 4, 0, 1, 5], [0, 3, 5, 1, 2, 4]]
data_weighted = [[5, 0, 1, 2, 3, 4], [5, 2, 4, 0, 1, 3], [5, 0, 2, 1, 3, 4], [5, 0, 1, 2, 3, 4], [1, 2, 4, 5, 3, 0], [1, 2, 0, 5, 3, 4], [5, 0, 2, 3, 4, 1], [0, 2, 1, 4, 3, 5], [0, 1, 2, 4, 5, 3], [5, 4, 0, 1, 2, 3], [5, 0, 1, 2, 3, 4], [5, 0, 1, 2, 3, 4], [1, 5, 2, 3, 4, 0], [0, 4, 5, 2, 1, 3], [2, 0, 1, 3, 4, 5], [2, 0, 1, 3, 4, 5], [4, 0, 3, 2, 5, 1], [0, 2, 5, 1, 4, 3], [0, 1, 3, 4, 5, 2], [5, 0, 1, 2, 3, 4], [5, 0, 1, 2, 3, 4], [2, 0, 1, 3, 4, 5], [3, 4, 0, 1, 2, 5], [0, 1, 2, 3, 4, 5], [0, 3, 4, 5, 1, 2], [0, 2, 4, 5, 1, 3], [2, 0, 1, 3, 5, 4], [5, 0, 1, 2, 3, 4], [0, 1, 2, 3, 4, 5], [2, 0, 3, 4, 1, 5], [4, 1, 2, 0, 3, 5], [2, 3, 4, 1, 0, 5], [1, 2, 3, 0, 5, 4], [1, 2, 3, 4, 5, 0], [0, 1, 2, 4, 5, 3], [0, 1, 2, 3, 4, 5], [0, 1, 2, 3, 4, 5], [0, 3, 4, 1, 2, 5], [0, 1, 3, 4, 2, 5], [0, 4, 2, 3, 1, 5], [0, 1, 3, 4, 5, 2], [0, 2, 3, 5, 1, 4], [0, 1, 2, 3, 4, 5], [2, 0, 1, 4, 3, 5], [0, 1, 3, 2, 4, 5], [1, 4, 2, 3, 0, 5], [0, 2, 3, 4, 1, 5], [2, 4, 0, 3, 1, 5], [0, 1, 5, 2, 3, 4]]

plot_sensitivity_bar(data_map, data_weighted, labels_colunas)